# Simple linear regression with no tuning

In [44]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

from pathlib import Path

In [ ]:
def preprocess_train_data(data_path, drop_std=True):
    """
    Reads in the training data, drops unnecessary columns, and creates the binary class2 column.
    If drop_std is True, also drops all columns ending with '.std'.
    Returns the processed DataFrame.
    """
    df = pd.read_csv(data_path)
    df = df.drop(columns=['id', 'partlybad','date'])
    df['class2'] = np.where(df['class4'].eq('nonevent'), 'nonevent', 'event')

    if drop_std:
        std_cols = [col for col in df.columns if col.endswith(".std")]
        df = df.drop(columns=std_cols)

    return df  

def split_xy(df):
    """
    Splits the DataFrame into features (X) and target variables (y2 for class2 and y4 for class4).
    Returns X, y2, and y4.
    """
    X = df.drop(columns=['class4', 'class2'])
    y2 = df['class2']
    y4 = df['class4']
    return X, y2, y4

def generate_synthetic_data(train_df, num_datasets=10):
    """
    Takes in dataset with mean and std columns for each numerical feature.
    Generates synthetic datasets by adding normally distributed noise based on the std values.
    The classes of the synthetic datasets are the same as the original dataset.
    Returns a single DataFrame with all synthetic datasets concatenated.
    """
    mean_cols = [col for col in train_df.columns if col.endswith(".mean")]
    std_cols = [col for col in train_df.columns if col.endswith(".std")]
    df_means = train_df.drop(columns=std_cols)
    df_stds = train_df.drop(columns=mean_cols)
    synthetic_dfs = []
    for i in range(num_datasets):
        synthetic_df = df_means.copy()
        for col in mean_cols:
            std_col = col.replace(".mean", ".std")
            synthetic_df[col] += np.random.normal(0, df_stds[std_col])
        synthetic_dfs.append(synthetic_df)
    return pd.concat(synthetic_dfs, ignore_index=True)


def kaggle_scoring(
    test_df,
    model_class2,
    model_prob,
    model_class4,
    print_partial_scores=False,
):
    """
    Imitates the Kaggle scoring process.
    The function takes in the test dataset and the three models for binary classification,
    probability estimation, and multiclass classification.
    Rerturns the final score.
    """

    # Process the test dataset for scoring
    std_cols = [col for col in test_df.columns if col.endswith(".std")]
    test_df = test_df.drop(columns=std_cols)
    test_df["class2"] = np.where(
        test_df["class4"].eq("nonevent"), "nonevent", "event"
    )
    X_test = test_df.drop(columns=["class4", "class2"])
    y_test2 = test_df["class2"]
    y_test4 = test_df["class4"]

    # Calculate scores for each log_model
    pred_class2 = model_class2.predict(X_test)
    pred_prob = model_prob.predict_proba(X_test)[:, 1]
    pred_class4 = model_class4.predict(X_test)

    binary_accuracy = accuracy_score(y_test2, pred_class2)
    perplexity = np.exp(log_loss(y_test2, pred_prob))
    multiclass_accuracy = accuracy_score(y_test4, pred_class4)
    total_score = (
        1
        / 3
        * (
            binary_accuracy
            + multiclass_accuracy
            + np.max([0.0, np.min([1.0, 2.0 - perplexity])])
        )
    )

    if print_partial_scores:
        print(f"Binary accuracy: {binary_accuracy:.4f}")
        print(f"Multiclass accuracy: {multiclass_accuracy:.4f}")
        print(f"Model perplexity: {perplexity:.4f}")
    return total_score


def generate_output(
    final_test_df,
    model_prob,
    model_class4,
):
    """
    Takes in the testing dataframe and models
    Generates the output format for the kaggle competition
    """
    std_cols = [col for col in final_test_df.columns if col.endswith('.std')]
    final_test_X = final_test_df.drop(columns=std_cols)
    final_test_X = final_test_X.drop(columns=["id", "partlybad", "date"])
    

    out_df = final_test_df[["id"]].copy()
    out_df["class4"] = model_class4.predict(final_test_X)
    out_df["p"] = model_prob.predict_proba(final_test_X)[:, 0]
    return out_df




## Reading in the data

In [46]:
from sklearn.model_selection import train_test_split

data_dir = Path('data')
output_dir = Path('outputs')

train_df = preprocess_train_data(data_dir / 'train.csv')
final_test_df = pd.read_csv(data_dir / 'test.csv')

train_df, test_df = train_test_split(train_df, test_size=0.3, random_state=42)

#train_df = generate_synthetic_data(train_df, num_datasets=10)

## Splitting to x and y variables

In [47]:
X_train, y_train2, y_train4 = split_xy(train_df)
X_test, y_test2, y_test4 = split_xy(test_df)
X_test = X_test.drop(columns=[col for col in X_test.columns if col.endswith('.std')])

## Constructing event-nonevent classifier

In [48]:
#Fitting logistic regression to class2

from sklearn.preprocessing import StandardScaler

log_model = LogisticRegression(C=0.04, l1_ratio=1, solver='liblinear')
log_model.fit(X_train, y_train2)

from sklearn.metrics import classification_report
y_pred = log_model.predict(X_test)
print(classification_report(y_test2, y_pred))


              precision    recall  f1-score   support

       event       0.86      0.81      0.84        70
    nonevent       0.81      0.86      0.84        65

    accuracy                           0.84       135
   macro avg       0.84      0.84      0.84       135
weighted avg       0.84      0.84      0.84       135



/opt/homebrew/anaconda3/envs/ml26/lib/python3.14/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


## Constructing extremely simple random forest class4 classifier (not a good method)

In [49]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train4)
print(classification_report(y_test4, rf.predict(X_test)))

              precision    recall  f1-score   support

          II       0.40      0.49      0.44        35
          Ia       0.50      0.12      0.20         8
          Ib       0.44      0.30      0.36        27
    nonevent       0.79      0.88      0.83        65

    accuracy                           0.61       135
   macro avg       0.53      0.45      0.46       135
weighted avg       0.60      0.61      0.60       135



### Trying PCA and RF


### Calculating a estimated Kaggle score

In [ ]:
total_score = kaggle_scoring(test_df, log_model, log_model, rf, print_partial_scores=True)
print(f'Kaggle score: {total_score:.4f}')

Binary accuracy: 0.8370
Multiclass accuracy: 0.6148
Model perplexity: 1.4476
Kaggle score: 0.6681


### Forming the output df

In [53]:
out_df = generate_output(
    final_test_df,
    model_prob=log_model,
    model_class4=rf,
)
out_df.to_csv(output_dir / 'not_good_submission.csv', index=False)
out_df.head(10)

,id,class4,p
0,450,nonevent,0.703935
1,451,Ib,0.984040
2,452,nonevent,0.050602
3,453,nonevent,0.035953
4,454,Ib,0.792843
5,455,nonevent,0.240684
6,456,Ib,0.875690
7,457,Ib,0.961008
8,458,nonevent,0.213413
9,459,nonevent,0.069522
